# SHROOM RAG E5 — Module 2 Phase A

## Offline Candidate Retrieval

Το notebook υλοποιεί μόνο την offline παραγωγή candidates από το unlabeled
model-agnostic training corpus. Το dense retrieval χρησιμοποιεί bounded inputs
έως 256 tokens και ένα πλήρες, επαληθευμένο, resumable local cache.

Η προσέγγιση ονομάζεται **unlabeled-training-corpus RAG**. Δεν αποτελεί
external-knowledge RAG, επειδή η knowledge base προέρχεται από το training corpus
του SHROOM.

## Σκοπός και όρια

Συγκρίνουμε χωρίς επιλογή «καλύτερης» μεθόδου:

1. exact normalized term matching,
2. BM25 lexical retrieval,
3. dense semantic retrieval με το ήδη τοπικό E3 embedding model,
4. fixed Reciprocal Rank Fusion.

Για κάθε μέθοδο διατηρούμε χωριστά τα queries `Q_SRC` και `Q_SRC_TGT`. Η
`Q_SRC_TGT` είναι reference-assisted retrieval και όχι ανεξάρτητη external
validation. Τα πέντε γνωστά regression cases είναι diagnostics του Module 1 και
όχι ανεξάρτητο evaluation set.

## 1. Imports και deterministic settings

In [ ]:
import hashlib
import json
import os
import random
import re
import sys
sys.dont_write_bytecode = True
import unicodedata
from importlib.metadata import version
from pathlib import Path


# Προσαρμοσμένος κώδικας και ιστορικά τεκμήρια έχουν χωριστές ταυτότητες.
import importlib.util as _module2_import
import sys as _module2_sys
from pathlib import Path as _Module2Path
_module2_sys.dont_write_bytecode = True
_module2_code_root = _Module2Path(os.environ["SHROOM_MODULE2_CODE_ROOT"])
if not _module2_code_root.is_absolute() or ".." in _module2_code_root.parts:
    raise ValueError("SHROOM_MODULE2_CODE_ROOT must be an absolute local directory.")
_module2_helper_path = _module2_code_root / "scripts" / "module2_dense_cache.py"
for _part in (_module2_helper_path, *_module2_helper_path.parents):
    if _part.is_symlink() or (getattr(_part.lstat(), "st_file_attributes", 0) & 0x400):
        raise ValueError("Reparse/symlink helper paths are not accepted.")
_module2_raw = _module2_helper_path.read_bytes()
if hashlib.sha256(_module2_raw).hexdigest() != os.environ["SHROOM_MODULE2_HELPER_SHA256"].lower():
    raise ValueError("Adapted helper pin mismatch; no fallback to historical source.")
if "module2_dense_cache" in _module2_sys.modules:
    _module2 = _module2_sys.modules["module2_dense_cache"]
    if _Module2Path(_module2.__file__) != _module2_helper_path or getattr(_module2, "_notebook_pin", None) != hashlib.sha256(_module2_raw).hexdigest():
        raise ValueError("A different/stale helper is already loaded; restart the kernel.")
else:
    _spec = _module2_import.spec_from_file_location("module2_dense_cache", _module2_helper_path)
    _module2 = _module2_import.module_from_spec(_spec)
    _module2_sys.modules["module2_dense_cache"] = _module2
    _spec.loader.exec_module(_module2)
    _module2._notebook_pin = hashlib.sha256(_module2_raw).hexdigest()
CTX = _module2.context_from_environment(require_cache=True)
print("Adapted Module 2: H0 evidence guards retained; runtime reproduction NOT VERIFIED.")

_module2._load_runtime_dependencies()
np, pd, torch = _module2.np, _module2.pd, _module2.torch
from IPython.display import display
from rank_bm25 import BM25Okapi

RANDOM_STATE = 42
PROTOTYPE_N = 20
TOP_K = 5
ANNOTATION_TOP_K = 3
RRF_CONSTANT = 60
QUERY_VARIANTS = ("Q_SRC", "Q_SRC_TGT")
RETRIEVAL_METHODS = ("exact", "bm25", "dense", "rrf")

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(True, warn_only=True)

assert os.sys.version.split()[0] == "3.11.9"
assert not torch.cuda.is_available()
print("RANDOM_STATE:", RANDOM_STATE)
print("Python:", os.sys.version.split()[0])
print("rank-bm25:", version("rank-bm25"))
print("CPU execution: True")
print("CUDA available:", torch.cuda.is_available())

## 2. Paths και απομόνωση outputs

In [ ]:
PROJECT_ROOT = CTX.input_root
RAG_DIR = CTX.evidence("RAG")
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = CTX.run_dir
DENSE_CACHE_DIR = CTX.cache_dir
SCRIPTS_DIR = _module2_code_root / "scripts"

from module2_dense_cache import (
    DENSE_CHUNK_SIZE,
    DENSE_MAX_SEQ_LENGTH,
    DENSE_TEXT_VERSION,
    EMBEDDING_DEVICE,
    EMBEDDING_DIMENSION as EXPECTED_EMBEDDING_DIMENSION,
    EMBEDDING_MODEL_ID,
    NORMALIZE_EMBEDDINGS,
    build_bounded_query_text,
    build_candidate_dense_records,
    compatibility_fields,
    dense_text_diagnostics,
    encode_texts,
    format_query,
    load_local_model,
    load_local_tokenizer,
    load_manifest,
    sha256_json,
    validate_and_load_complete_cache,
)

TRAIN_PATH = (
    DATA_DIR / "Unlabeled train" / "SHROOM_unlabeled-training-data-v2"
    / "train.model-agnostic.json"
)
DEV_PATH = DATA_DIR / "Dev" / "SHROOM_dev-v2" / "val.model-agnostic.json"

_module2._physical_path(TRAIN_PATH)
_module2._physical_path(DEV_PATH)
MODULE1_TRIAL_PATH = RAG_DIR / "shroom_rag_e5_module1_gemini_trial.ipynb"
MODULE1_CACHE_PATH = RAG_DIR / "results" / "rag_module1" / "module1_gemini_cache.jsonl"
EXPECTED_MODULE1_TRIAL_SHA256 = "A7E0455A11EFFB7C195EFFB1CACB542DD82992FCCE027FB0D297BE598557D20B"
EXPECTED_MODULE1_CACHE_SHA256 = "A942A0CC228DEF64B61389BCF7B7A5CB0A93BBCC7B0AA1D726826A9993B14CA5"

MANIFEST_PATH = OUTPUT_DIR / "module2_input_manifest.json"
CANDIDATES_PATH = OUTPUT_DIR / "module2_candidates_top5.jsonl"
ANNOTATIONS_PATH = OUTPUT_DIR / "module2_relevance_annotations.csv"
RESULTS_REPORT_PATH = OUTPUT_DIR / "MODULE2_PHASE_A_RESULTS.md"

assert OUTPUT_DIR == CTX.run_dir
for output_path in (MANIFEST_PATH, CANDIDATES_PATH, ANNOTATIONS_PATH, RESULTS_REPORT_PATH):
    assert output_path.parent == OUTPUT_DIR
print("Module 2 output isolation:", OUTPUT_DIR.resolve())

## 3. Dataset και leakage guards

In [ ]:
FORBIDDEN_FIELDS = {
    "label",
    "labels",
    "label_bin",
    "p(Hallucination)",
    "p_hallucination",
}


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


def assert_no_official_test_path(*paths: Path) -> None:
    for path in paths:
        normalized = str(path).replace("\\", "/").lower()
        assert "/test" not in normalized and "test.model-" not in normalized, path


assert_no_official_test_path(TRAIN_PATH, DEV_PATH)
assert sha256_file(MODULE1_TRIAL_PATH) == EXPECTED_MODULE1_TRIAL_SHA256
assert sha256_file(MODULE1_CACHE_PATH) == EXPECTED_MODULE1_CACHE_SHA256
assert MANIFEST_PATH.parent == OUTPUT_DIR
assert CANDIDATES_PATH.parent == OUTPUT_DIR
assert ANNOTATIONS_PATH.parent == OUTPUT_DIR

print("Official-test path guard passed.")
print("Locked Module 1 hashes passed.")

## 4. Φόρτωση επιτρεπόμενων inputs

In [ ]:
def load_json_dataframe(path: Path) -> pd.DataFrame:
    with path.open("r", encoding="utf-8") as stream:
        return pd.DataFrame(json.load(stream))


train_df = load_json_dataframe(TRAIN_PATH)
development_full_df = load_json_dataframe(DEV_PATH)

required_columns = {"task", "src", "tgt", "hyp"}
assert required_columns.issubset(train_df.columns)
assert required_columns.issubset(development_full_df.columns)
assert not FORBIDDEN_FIELDS & set(train_df.columns)
assert len(train_df) == 30000
assert len(development_full_df) == 499

# Από εδώ και μετά το development dataframe δεν περιέχει gold fields.
development_inputs_df = development_full_df[["task", "src", "tgt", "hyp"]].copy()
del development_full_df
assert not FORBIDDEN_FIELDS & set(development_inputs_df.columns)

print("Unlabeled train rows:", len(train_df))
print("Development input rows:", len(development_inputs_df))
print("Forbidden fields in retrieval inputs:", sorted(FORBIDDEN_FIELDS & set(development_inputs_df.columns)))

## 5. Ανακατασκευή της DM knowledge base

In [ ]:
CLOSED_DEFINE_PATTERN = re.compile(
    r"<define>\s*(.*?)\s*</define>",
    flags=re.IGNORECASE | re.DOTALL,
)
OPEN_DEFINE_MARKER = re.compile(r"<define>", flags=re.IGNORECASE)


def clean_text(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def extract_define_term(source_text) -> str | None:
    text = clean_text(source_text)
    if not text:
        return None
    closed_match = CLOSED_DEFINE_PATTERN.search(text)
    if closed_match:
        candidate = closed_match.group(1).strip()
        return candidate or None
    parts = OPEN_DEFINE_MARKER.split(text)
    if len(parts) >= 3:
        candidate = parts[1].strip()
        return candidate or None
    return None


def normalize_term(term) -> str | None:
    text = clean_text(term)
    if not text:
        return None
    text = unicodedata.normalize("NFKC", text).lower()
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s*-\s*", "-", text)
    text = re.sub(r"[^\w\s\-']+", " ", text, flags=re.UNICODE)
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def valid_definition(value) -> bool:
    text = clean_text(value)
    return bool(text and re.search(r"[A-Za-z0-9]", text))


def build_dm_knowledge_base(frame: pd.DataFrame) -> pd.DataFrame:
    dm = frame.loc[frame["task"].astype(str).eq("DM")].copy()
    dm = dm.reset_index().rename(columns={"index": "evidence_id"})
    dm["term_raw"] = dm["src"].map(extract_define_term)
    dm["term_key"] = dm["term_raw"].map(normalize_term)
    dm["definition"] = dm["tgt"].map(clean_text)
    dm["source_context"] = dm["src"].map(clean_text)
    knowledge_base = dm.loc[
        dm["term_key"].notna() & dm["definition"].map(valid_definition),
        ["evidence_id", "term_raw", "term_key", "definition", "source_context"],
    ].copy()
    return knowledge_base.drop_duplicates(
        subset=["term_key", "definition"]
    ).reset_index(drop=True)


assert extract_define_term("A <define> term </define> here") == "term"
assert extract_define_term("A <define> term <define> here") == "term"
assert normalize_term(" Sheath - Knife ") == "sheath knife"

dm_kb = build_dm_knowledge_base(train_df)
assert int(train_df["task"].eq("DM").sum()) == 10000
assert len(dm_kb) == 9577
assert dm_kb["evidence_id"].is_unique
assert dm_kb["evidence_id"].tolist() == sorted(dm_kb["evidence_id"].tolist())

print("DM training rows:", int(train_df["task"].eq("DM").sum()))
print("Knowledge-base rows:", len(dm_kb))
print("Unique normalized terms:", dm_kb["term_key"].nunique())

## 6. Ίδιο σταθερό sample 20 examples

In [ ]:
dm_development = development_inputs_df.loc[
    development_inputs_df["task"].astype(str).eq("DM")
].copy()
dm_development = dm_development.reset_index().rename(
    columns={"index": "development_index"}
)

prototype_df = (
    dm_development.sample(n=PROTOTYPE_N, random_state=RANDOM_STATE)
    .sort_values("development_index")
    .reset_index(drop=True)
)
prototype_df["define_term"] = prototype_df["src"].map(extract_define_term)
prototype_df["define_term_key"] = prototype_df["define_term"].map(normalize_term)

EXPECTED_SAMPLE_INDICES = [
    15, 16, 19, 30, 42, 51, 55, 75, 78, 85,
    122, 137, 154, 158, 161, 165, 166, 175, 177, 185,
]
assert prototype_df["development_index"].astype(int).tolist() == EXPECTED_SAMPLE_INDICES
assert prototype_df["define_term"].notna().all()
assert not FORBIDDEN_FIELDS & set(prototype_df.columns)

print("Prototype rows:", len(prototype_df))
print("Sample indices:", EXPECTED_SAMPLE_INDICES)

## 7. Input manifests και hashes

In [ ]:
def canonical_records_hash(frame: pd.DataFrame, columns: list[str]) -> str:
    records = frame[columns].to_dict(orient="records")
    payload = json.dumps(
        records,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest().upper()


KB_HASH = canonical_records_hash(
    dm_kb,
    ["evidence_id", "term_raw", "term_key", "definition", "source_context"],
)
SAMPLE_HASH = canonical_records_hash(
    prototype_df,
    ["development_index", "task", "src", "tgt", "hyp", "define_term", "define_term_key"],
)
TRAIN_FILE_HASH = sha256_file(TRAIN_PATH)
DEV_FILE_HASH = sha256_file(DEV_PATH)

print("Knowledge-base SHA-256:", KB_HASH)
print("Sample SHA-256:", SAMPLE_HASH)

## 8. Bounded query construction και token diagnostics

Το term διατηρείται πλήρες. Για το `Q_SRC_TGT` η primary reference διατηρείται
πριν δοθεί το υπόλοιπο token budget στο source context. Έτσι παραμένει ρητά
reference-assisted retrieval. Κανένα field δεν αφήνεται σε σιωπηλό truncation.

In [ ]:
dense_tokenizer = load_local_tokenizer()

query_records = []
query_dense_records = []
for _, row in prototype_df.iterrows():
    for variant in QUERY_VARIANTS:
        reference = None if variant == "Q_SRC" else clean_text(row["tgt"])
        dense_record = build_bounded_query_text(
            dense_tokenizer,
            clean_text(row["define_term"]),
            clean_text(row["src"]),
            reference,
        )
        query_dense_records.append(dense_record)
        query_records.append({
            "development_index": int(row["development_index"]),
            "term": clean_text(row["define_term"]),
            "term_key": clean_text(row["define_term_key"]),
            "query_variant": variant,
            "query_text_unbounded": format_query(
                clean_text(row["define_term"]),
                clean_text(row["src"]),
                reference,
            ),
            "query_text": dense_record.text,
            "unbounded_token_length": dense_record.unbounded_token_length,
            "bounded_token_length": dense_record.bounded_token_length,
            "context_truncated": dense_record.context_truncated,
            "reference_truncated": dense_record.reference_truncated,
            "context_fallback": dense_record.context_fallback,
        })

query_df = pd.DataFrame(query_records)
QUERY_TOKEN_DIAGNOSTICS = dense_text_diagnostics(query_dense_records)
assert len(query_df) == PROTOTYPE_N * len(QUERY_VARIANTS) == 40
assert not FORBIDDEN_FIELDS & set(query_df.columns)
assert query_df["bounded_token_length"].le(DENSE_MAX_SEQ_LENGTH).all()

for development_index, group in query_df.groupby("development_index"):
    assert group["query_text"].nunique() == 2
    assert set(group["query_variant"]) == set(QUERY_VARIANTS)

# Ίδια ανακατασκευή για deterministic equality.
rebuilt_queries = []
for _, row in prototype_df.iterrows():
    for variant in QUERY_VARIANTS:
        reference = None if variant == "Q_SRC" else clean_text(row["tgt"])
        rebuilt_queries.append(
            build_bounded_query_text(
                dense_tokenizer,
                clean_text(row["define_term"]),
                clean_text(row["src"]),
                reference,
            ).text
        )
assert rebuilt_queries == query_df["query_text"].tolist()

print("Query rows:", len(query_df))
print("Distinct query variants:", sorted(query_df["query_variant"].unique()))
display(pd.DataFrame({
    "unbounded": QUERY_TOKEN_DIAGNOSTICS["unbounded_token_lengths"],
    "bounded": QUERY_TOKEN_DIAGNOSTICS["bounded_token_lengths"],
}))
print("Query context-only truncations:", QUERY_TOKEN_DIAGNOSTICS["context_only_truncated_count"])
print("Query reference truncations:", QUERY_TOKEN_DIAGNOSTICS["reference_truncated_count"])
print("Query context fallbacks:", QUERY_TOKEN_DIAGNOSTICS["context_fallback_counts"])

## 9. Exact retrieval baseline

In [ ]:
evidence_ids = dm_kb["evidence_id"].astype(int).to_numpy()
kb_by_id = dm_kb.set_index("evidence_id", drop=False)
term_to_positions = {
    term_key: np.asarray(indices, dtype=int)
    for term_key, indices in dm_kb.groupby("term_key", sort=False).indices.items()
}


def stable_top_positions(scores: np.ndarray, top_k: int = TOP_K) -> np.ndarray:
    numeric_scores = np.asarray(scores, dtype=float)
    assert numeric_scores.shape == evidence_ids.shape
    order = np.lexsort((evidence_ids, -numeric_scores))
    return order[:top_k]


def exact_top5(term_key: str) -> list[dict]:
    positions = term_to_positions.get(term_key, np.asarray([], dtype=int))
    positions = sorted(positions.tolist(), key=lambda pos: int(evidence_ids[pos]))[:TOP_K]
    return [
        {
            "evidence_id": int(evidence_ids[position]),
            "score": 1.0,
            "rank": rank,
        }
        for rank, position in enumerate(positions, start=1)
    ]


rankings = {}
score_cache = {}
for query in query_df.itertuples(index=False):
    key = (int(query.development_index), query.query_variant)
    rankings[key] = {"exact": exact_top5(query.term_key)}
    score_cache[key] = {}

exact_nonempty = sum(bool(methods["exact"]) for methods in rankings.values())
print("Exact non-empty query lists:", exact_nonempty, "/", len(rankings))

## 10. BM25 lexical retrieval

Το BM25 χρησιμοποιεί τα ίδια bounded query strings. Η candidate lexical
representation παραμένει ξεχωριστή από τη versioned dense representation.

In [ ]:
TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)


def lexical_tokens(text: str) -> list[str]:
    normalized = unicodedata.normalize("NFKC", clean_text(text)).lower()
    return TOKEN_PATTERN.findall(normalized)


def bm25_candidate_text(row: pd.Series) -> str:
    return (
        f"TERM: {clean_text(row['term_raw'])}\n"
        f"DEFINITION: {clean_text(row['definition'])}\n"
        f"SOURCE CONTEXT: {clean_text(row['source_context'])}"
    )


bm25_candidate_texts = [bm25_candidate_text(row) for _, row in dm_kb.iterrows()]
candidate_tokens = [lexical_tokens(text) for text in bm25_candidate_texts]
bm25_index = BM25Okapi(candidate_tokens)


def bm25_top5(query_text: str) -> tuple[list[dict], np.ndarray]:
    scores = np.asarray(bm25_index.get_scores(lexical_tokens(query_text)), dtype=float)
    positions = stable_top_positions(scores)
    results = [
        {
            "evidence_id": int(evidence_ids[position]),
            "score": float(scores[position]),
            "rank": rank,
        }
        for rank, position in enumerate(positions, start=1)
    ]
    return results, scores


for query in query_df.itertuples(index=False):
    key = (int(query.development_index), query.query_variant)
    results, scores = bm25_top5(query.query_text)
    rankings[key]["bm25"] = results
    score_cache[key]["bm25"] = scores

print("BM25 lists:", len(rankings))
print("BM25 candidates per list:", TOP_K)

## 11. Dense retrieval από πλήρες resumable cache

Τα 9.577 candidate embeddings δεν επανυπολογίζονται εδώ. Το cell επαληθεύει
όλα τα compatibility fields και φορτώνει μόνο ολοκληρωμένα atomic chunks. Στη
συνέχεια κωδικοποιεί τα 40 bounded queries με το ίδιο local CPU model.

In [ ]:
dense_candidate_records = build_candidate_dense_records(dense_tokenizer, dm_kb)
dense_candidate_texts = [record.text for record in dense_candidate_records]
CANDIDATE_TOKEN_DIAGNOSTICS = dense_text_diagnostics(dense_candidate_records)
assert CANDIDATE_TOKEN_DIAGNOSTICS["all_bounded_at_most_256"]

dense_cache_manifest = load_manifest(DENSE_CACHE_DIR)
selected_batch_size = int(dense_cache_manifest["selected_batch_size"])
embedding_model = load_local_model()

kb_hashes_for_cache = {
    "ordered_kb_sha256": KB_HASH,
    "ordered_candidate_ids_sha256": sha256_json(
        dm_kb["evidence_id"].astype(int).tolist()
    ),
}
expected_cache_compatibility = compatibility_fields(
    dm_kb,
    kb_hashes_for_cache,
    dense_candidate_texts,
    embedding_model,
    selected_batch_size,
)
candidate_embeddings, dense_cache_manifest = validate_and_load_complete_cache(
    DENSE_CACHE_DIR,
    expected_cache_compatibility,
    evidence_ids.astype(np.int64),
)
query_embeddings = encode_texts(
    embedding_model,
    query_df["query_text"].tolist(),
    selected_batch_size,
)

EMBEDDING_DIMENSION = int(candidate_embeddings.shape[1])
DENSE_AVAILABLE = True
DENSE_BLOCKER = None
assert candidate_embeddings.shape == (9577, EXPECTED_EMBEDDING_DIMENSION)
assert query_embeddings.shape == (40, EXPECTED_EMBEDDING_DIMENSION)
assert np.isfinite(candidate_embeddings).all()
assert np.isfinite(query_embeddings).all()
assert np.allclose(np.linalg.norm(candidate_embeddings, axis=1), 1.0, atol=1e-5)
assert np.allclose(np.linalg.norm(query_embeddings, axis=1), 1.0, atol=1e-5)

# Μικρό deterministic cached-subset rerun, χωρίς νέο full-corpus encoding.
subset_positions = np.asarray([0, 1, 2, 3], dtype=int)
subset_repeat = encode_texts(
    embedding_model,
    [dense_candidate_texts[position] for position in subset_positions],
    selected_batch_size,
)
assert np.allclose(
    subset_repeat,
    candidate_embeddings[subset_positions],
    atol=1e-6,
)

for query_position, query in enumerate(query_df.itertuples(index=False)):
    key = (int(query.development_index), query.query_variant)
    scores = candidate_embeddings @ query_embeddings[query_position]
    positions = stable_top_positions(scores)
    rankings[key]["dense"] = [
        {
            "evidence_id": int(evidence_ids[position]),
            "score": float(scores[position]),
            "rank": rank,
        }
        for rank, position in enumerate(positions, start=1)
    ]
    score_cache[key]["dense"] = np.asarray(scores, dtype=float)

print("Dense available:", DENSE_AVAILABLE)
print("Embedding model:", EMBEDDING_MODEL_ID)
print("Embedding dimension:", EMBEDDING_DIMENSION)
print("Normalization:", NORMALIZE_EMBEDDINGS)
print("Device:", EMBEDDING_DEVICE)
print("Dense text version:", DENSE_TEXT_VERSION)
print("Max sequence length:", DENSE_MAX_SEQ_LENGTH)
print("Selected batch size:", selected_batch_size)
print("Chunk size:", DENSE_CHUNK_SIZE)
display(pd.DataFrame({
    "unbounded": CANDIDATE_TOKEN_DIAGNOSTICS["unbounded_token_lengths"],
    "bounded": CANDIDATE_TOKEN_DIAGNOSTICS["bounded_token_lengths"],
}))
print("Candidate context-only truncations:", CANDIDATE_TOKEN_DIAGNOSTICS["context_only_truncated_count"])
print("Definition truncations:", CANDIDATE_TOKEN_DIAGNOSTICS["definition_truncated_count"])
print("Candidate context fallbacks:", CANDIDATE_TOKEN_DIAGNOSTICS["context_fallback_counts"])

## 12. Fixed Reciprocal Rank Fusion

In [ ]:
def reciprocal_rank_fusion(method_rankings: dict[str, list[dict]]) -> list[dict]:
    fused = {}
    for method in ("exact", "bm25", "dense"):
        for item in method_rankings.get(method, []):
            evidence_id = int(item["evidence_id"])
            entry = fused.setdefault(
                evidence_id,
                {"evidence_id": evidence_id, "score": 0.0, "methods": []},
            )
            entry["score"] += 1.0 / (RRF_CONSTANT + int(item["rank"]))
            entry["methods"].append(method)

    ordered = sorted(
        fused.values(),
        key=lambda item: (-float(item["score"]), int(item["evidence_id"])),
    )
    return [
        {
            **item,
            "rank": rank,
            "methods": sorted(item["methods"]),
        }
        for rank, item in enumerate(ordered, start=1)
    ]


for key, methods in rankings.items():
    methods["rrf_all"] = reciprocal_rank_fusion(methods)
    methods["rrf"] = methods["rrf_all"][:TOP_K]

print("RRF constant:", RRF_CONSTANT)
print("RRF lists:", len(rankings))

## 13. Candidate deduplication και provenance

In [ ]:
def rank_map(items: list[dict]) -> dict[int, int]:
    return {int(item["evidence_id"]): int(item["rank"]) for item in items}


def rrf_score_map(items: list[dict]) -> dict[int, float]:
    return {int(item["evidence_id"]): float(item["score"]) for item in items}


candidate_records = []
for query in query_df.itertuples(index=False):
    key = (int(query.development_index), query.query_variant)
    method_lists = rankings[key]
    exact_ranks = rank_map(method_lists["exact"])
    bm25_ranks = rank_map(method_lists["bm25"])
    dense_ranks = rank_map(method_lists["dense"])
    rrf_ranks = rank_map(method_lists["rrf_all"])
    rrf_scores = rrf_score_map(method_lists["rrf_all"])
    method_sets = {}
    for method in ("exact", "bm25", "dense"):
        for item in method_lists[method]:
            method_sets.setdefault(int(item["evidence_id"]), set()).add(method)

    for output_method in RETRIEVAL_METHODS:
        for output_item in method_lists[output_method]:
            evidence_id = int(output_item["evidence_id"])
            candidate = kb_by_id.loc[evidence_id]
            position = int(dm_kb.index[dm_kb["evidence_id"].eq(evidence_id)][0])
            bm25_score = float(score_cache[key]["bm25"][position])
            dense_score = (
                float(score_cache[key]["dense"][position])
                if DENSE_AVAILABLE else None
            )
            candidate_records.append({
                "development_index": int(query.development_index),
                "query_variant": query.query_variant,
                "retrieval_method": output_method,
                "method_rank": int(output_item["rank"]),
                "evidence_id": evidence_id,
                "normalized_term": clean_text(candidate["term_key"]),
                "raw_term": clean_text(candidate["term_raw"]),
                "definition": clean_text(candidate["definition"]),
                "source_context": clean_text(candidate["source_context"]),
                "exact_match": bool(candidate["term_key"] == query.term_key),
                "retrieval_methods": sorted(method_sets.get(evidence_id, set())),
                "exact_rank": exact_ranks.get(evidence_id),
                "bm25_score": bm25_score,
                "bm25_rank": bm25_ranks.get(evidence_id),
                "dense_cosine_score": dense_score,
                "dense_rank": dense_ranks.get(evidence_id),
                "rrf_score": rrf_scores.get(evidence_id),
                "rrf_rank": rrf_ranks.get(evidence_id),
            })

candidate_df = pd.DataFrame(candidate_records)
assert not candidate_df.empty
assert not FORBIDDEN_FIELDS & set(candidate_df.columns)

duplicate_counts = candidate_df.groupby(
    ["development_index", "query_variant", "retrieval_method", "evidence_id"]
).size()
assert int(duplicate_counts.max()) == 1

print("Candidate artifact rows:", len(candidate_df))
display(
    candidate_df.groupby(["query_variant", "retrieval_method"])
    .size().rename("candidate_rows").reset_index()
)

## 14. Regression-case previews

In [ ]:
REGRESSION_INDICES = [30, 55, 122, 161, 185]
REGRESSION_TERMS = {
    30: "preciosity",
    55: "dateliner",
    122: "continuous",
    161: "weary",
    185: "inconsistent",
}

regression_preview_df = candidate_df.loc[
    candidate_df["development_index"].isin(REGRESSION_INDICES)
    & candidate_df["retrieval_method"].eq("rrf")
    & candidate_df["method_rank"].le(3),
    [
        "development_index", "query_variant", "method_rank", "evidence_id",
        "raw_term", "definition", "exact_match", "retrieval_methods",
        "bm25_score", "dense_cosine_score", "rrf_score",
    ],
].copy()
regression_preview_df.insert(
    1,
    "diagnostic_term",
    regression_preview_df["development_index"].map(REGRESSION_TERMS),
)

print("Regression cases are diagnostics, not an independent evaluation set.")
display(regression_preview_df)

## 15. Annotation artifact generation

In [ ]:
annotation_columns = [
    "development_index", "term", "query_variant", "rank", "evidence_id",
    "definition", "candidate_context", "exact_match", "retrieval_methods",
    "exact_rank", "bm25_score", "bm25_rank", "dense_cosine_score",
    "dense_rank", "rrf_score", "rrf_rank", "relevance_category",
    "annotation_notes",
]

annotation_records = []
rrf_top3 = candidate_df.loc[
    candidate_df["retrieval_method"].eq("rrf")
    & candidate_df["method_rank"].le(ANNOTATION_TOP_K)
].copy()

term_lookup = prototype_df.set_index("development_index")["define_term"].to_dict()
for row in rrf_top3.itertuples(index=False):
    annotation_records.append({
        "development_index": int(row.development_index),
        "term": clean_text(term_lookup[int(row.development_index)]),
        "query_variant": row.query_variant,
        "rank": int(row.method_rank),
        "evidence_id": int(row.evidence_id),
        "definition": clean_text(row.definition),
        "candidate_context": clean_text(row.source_context),
        "exact_match": bool(row.exact_match),
        "retrieval_methods": json.dumps(row.retrieval_methods, ensure_ascii=False),
        "exact_rank": row.exact_rank,
        "bm25_score": row.bm25_score,
        "bm25_rank": row.bm25_rank,
        "dense_cosine_score": row.dense_cosine_score,
        "dense_rank": row.dense_rank,
        "rrf_score": row.rrf_score,
        "rrf_rank": row.rrf_rank,
        "relevance_category": "",
        "annotation_notes": "",
    })

annotation_df = pd.DataFrame(annotation_records, columns=annotation_columns)
assert annotation_df["relevance_category"].fillna("").eq("").all()
assert annotation_df["annotation_notes"].fillna("").eq("").all()
assert not FORBIDDEN_FIELDS & set(annotation_df.columns)

print("Annotation shortlist rows:", len(annotation_df))
print("Annotations completed automatically: 0")

## 16. Offline tests και verification summary

Οι έλεγχοι καλύπτουν cache compatibility, shapes, normalization, bounded token
lengths, deterministic subset rerun, rankings, leakage και output isolation.

In [ ]:
def assert_unique_ranked_lists() -> None:
    for key, methods in rankings.items():
        for method in RETRIEVAL_METHODS:
            ids = [int(item["evidence_id"]) for item in methods[method]]
            assert len(ids) == len(set(ids)), (key, method)
            ranks = [int(item["rank"]) for item in methods[method]]
            assert ranks == list(range(1, len(ranks) + 1)), (key, method)
            if method in {"bm25", "dense", "rrf"}:
                assert len(ids) == TOP_K, (key, method)


def assert_scores() -> None:
    assert np.isfinite(candidate_df["bm25_score"].astype(float)).all()
    assert candidate_df["bm25_score"].astype(float).ge(0).all()
    dense_values = candidate_df["dense_cosine_score"].dropna().astype(float)
    assert np.isfinite(dense_values).all()
    assert dense_values.between(-1.000001, 1.000001).all()
    rrf_values = candidate_df["rrf_score"].dropna().astype(float)
    assert np.isfinite(rrf_values).all()
    assert rrf_values.gt(0).all()


def rerun_first_rankings() -> None:
    first_query = query_df.iloc[0]
    key = (int(first_query["development_index"]), first_query["query_variant"])
    bm25_repeat, _ = bm25_top5(first_query["query_text"])
    assert bm25_repeat == rankings[key]["bm25"]
    assert reciprocal_rank_fusion(rankings[key])[:TOP_K] == rankings[key]["rrf"]
    repeated_query = encode_texts(
        embedding_model,
        [first_query["query_text"]],
        selected_batch_size,
    )[0]
    repeated_scores = candidate_embeddings @ repeated_query
    positions = stable_top_positions(repeated_scores)
    repeated_ids = [int(evidence_ids[position]) for position in positions]
    original_ids = [int(item["evidence_id"]) for item in rankings[key]["dense"]]
    assert repeated_ids == original_ids


# Stable tie-breaking: χαμηλότερο evidence_id προηγείται σε ίδιο score.
synthetic_scores = np.zeros_like(evidence_ids, dtype=float)
synthetic_scores[:2] = 1.0
synthetic_order = stable_top_positions(synthetic_scores, top_k=2)
assert evidence_ids[synthetic_order].tolist() == sorted(evidence_ids[:2].tolist())

assert_unique_ranked_lists()
assert_scores()
rerun_first_rankings()

required_candidate_fields = {
    "evidence_id", "normalized_term", "raw_term", "definition",
    "source_context", "exact_match", "retrieval_methods", "exact_rank",
    "bm25_score", "bm25_rank", "dense_cosine_score", "dense_rank",
    "rrf_score", "rrf_rank", "query_variant", "development_index",
}
assert required_candidate_fields.issubset(candidate_df.columns)
assert not FORBIDDEN_FIELDS & set(query_df.columns)
assert not FORBIDDEN_FIELDS & set(candidate_df.columns)
assert not FORBIDDEN_FIELDS & set(annotation_df.columns)
assert prototype_df["development_index"].astype(int).tolist() == EXPECTED_SAMPLE_INDICES
assert len(dm_kb) == 9577 and dm_kb["evidence_id"].is_unique
assert len(query_df) == 40
assert query_df.groupby("development_index")["query_text"].nunique().eq(2).all()
assert query_df["bounded_token_length"].le(DENSE_MAX_SEQ_LENGTH).all()
assert max(record.bounded_token_length for record in dense_candidate_records) <= DENSE_MAX_SEQ_LENGTH
assert dense_cache_manifest["cache_complete"] is True
assert candidate_embeddings.shape == (9577, 768)
assert query_embeddings.shape == (40, 768)
assert sha256_file(MODULE1_TRIAL_PATH) == EXPECTED_MODULE1_TRIAL_SHA256
assert sha256_file(MODULE1_CACHE_PATH) == EXPECTED_MODULE1_CACHE_SHA256

TEST_SUMMARY = {
    "forbidden_fields_absent": True,
    "official_test_guard": True,
    "sample_indices_stable": True,
    "knowledge_base_stable": True,
    "forty_distinct_bounded_queries": True,
    "all_dense_inputs_at_most_256_tokens": True,
    "cache_compatible_and_complete": True,
    "candidate_embeddings_shape_normalized_finite": True,
    "query_embeddings_shape_normalized_finite": True,
    "cached_subset_deterministic": True,
    "bm25_ranking_deterministic": True,
    "dense_ranking_deterministic": True,
    "rrf_ranking_deterministic": True,
    "stable_tie_breaking": True,
    "no_duplicate_ranked_candidates": True,
    "scores_finite_and_in_range": True,
    "query_variants_distinct": True,
    "candidate_provenance_complete": True,
    "module2_output_isolation": True,
    "module1_hashes_unchanged": True,
}

print("Offline tests passed:", sum(TEST_SUMMARY.values()), "/", len(TEST_SUMMARY))
display(pd.DataFrame(TEST_SUMMARY.items(), columns=["test", "passed"]))

## 17. Atomic αποθήκευση artifacts και Phase A report

Τα artifacts γράφονται μόνο στο `results/rag_module2`. Το annotation CSV είναι
κενό template ως προς τις manual relevance στήλες και δεν υπολογίζονται
relevance metrics.

In [ ]:
def python_value(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    CTX.write_text(path, text, encoding)


manifest = {
    "phase": "Module 2 Phase A: Offline Candidate Retrieval",
    "rag_type": "unlabeled-training-corpus RAG",
    "random_state": RANDOM_STATE,
    "dataset_paths": {
        "unlabeled_train": str(TRAIN_PATH),
        "development_inputs": str(DEV_PATH),
    },
    "input_hashes": {
        "unlabeled_train_file_sha256": TRAIN_FILE_HASH,
        "development_file_sha256": DEV_FILE_HASH,
        "dm_knowledge_base_sha256": KB_HASH,
        "prototype_sample_sha256": SAMPLE_HASH,
    },
    "locked_module1_hashes": {
        "trial_notebook_sha256": EXPECTED_MODULE1_TRIAL_SHA256,
        "cache_sha256": EXPECTED_MODULE1_CACHE_SHA256,
    },
    "sample_indices": EXPECTED_SAMPLE_INDICES,
    "knowledge_base_rows": int(len(dm_kb)),
    "query_variants": list(QUERY_VARIANTS),
    "query_variant_note": {
        "Q_SRC": "term plus bounded source context",
        "Q_SRC_TGT": "reference-assisted: term plus bounded source context and primary reference",
    },
    "retrieval_configuration": {
        "top_k": TOP_K,
        "annotation_top_k": ANNOTATION_TOP_K,
        "rrf_constant": RRF_CONSTANT,
        "fusion_weights_tuned": False,
        "relevance_thresholds_defined": False,
    },
    "packages": {
        "python": os.sys.version.split()[0],
        "rank-bm25": version("rank-bm25"),
        "sentence-transformers": version("sentence-transformers"),
        "transformers": version("transformers"),
        "torch": version("torch"),
        "numpy": version("numpy"),
        "scikit-learn": version("scikit-learn"),
    },
    "embedding": {
        "model_identifier": EMBEDDING_MODEL_ID,
        "local_files_only": True,
        "available": True,
        "dimension": EMBEDDING_DIMENSION,
        "normalize_embeddings": NORMALIZE_EMBEDDINGS,
        "device": EMBEDDING_DEVICE,
        "deterministic_algorithms": True,
        "dense_text_version": DENSE_TEXT_VERSION,
        "max_seq_length": DENSE_MAX_SEQ_LENGTH,
        "selected_batch_size": selected_batch_size,
        "chunk_size": DENSE_CHUNK_SIZE,
        "dtype": "float32",
        "candidate_token_diagnostics": CANDIDATE_TOKEN_DIAGNOSTICS,
        "query_token_diagnostics": QUERY_TOKEN_DIAGNOSTICS,
        "cache_manifest": str(DENSE_CACHE_DIR / "dense_cache_manifest.json"),
        "cache_builder_launches": int(dense_cache_manifest["builder_launches"]),
        "cache_resumptions": int(dense_cache_manifest["cache_resumptions"]),
        "cache_chunks": len(dense_cache_manifest["completed_chunk_ranges"]),
        "total_chunk_encoding_seconds": float(dense_cache_manifest["total_chunk_encoding_seconds"]),
        "benchmark_results": dense_cache_manifest["benchmark_results"],
    },
}

manifest_text = json.dumps(manifest, ensure_ascii=False, indent=2) + "\n"
atomic_write_text(MANIFEST_PATH, manifest_text)

candidate_lines = []
for record in candidate_df.to_dict(orient="records"):
    clean_record = {key: python_value(value) for key, value in record.items()}
    candidate_lines.append(json.dumps(clean_record, ensure_ascii=False))
atomic_write_text(CANDIDATES_PATH, "\n".join(candidate_lines) + "\n")

annotation_text = annotation_df.to_csv(index=False)
atomic_write_text(ANNOTATIONS_PATH, annotation_text, encoding="utf-8-sig")

candidate_counts = (
    candidate_df.groupby(["query_variant", "retrieval_method"])
    .size().rename("rows").reset_index()
)
exact_coverage = {
    variant: sum(
        bool(rankings[(int(index), variant)]["exact"])
        for index in EXPECTED_SAMPLE_INDICES
    ) / len(EXPECTED_SAMPLE_INDICES)
    for variant in QUERY_VARIANTS
}

count_lines = [
    f"| `{row.query_variant}` | `{row.retrieval_method}` | {int(row.rows)} |"
    for row in candidate_counts.itertuples(index=False)
]

preview_lines = []
for row in regression_preview_df.itertuples(index=False):
    definition = clean_text(row.definition).replace("|", "\\|")
    if len(definition) > 120:
        definition = definition[:117] + "..."
    methods = ", ".join(row.retrieval_methods)
    preview_lines.append(
        f"| {int(row.development_index)} | `{row.diagnostic_term}` | "
        f"`{row.query_variant}` | {int(row.method_rank)} | "
        f"{int(row.evidence_id)} | {row.raw_term} | {methods} | {definition} |"
    )


def stats_line(name: str, diagnostics: dict, field: str) -> str:
    values = diagnostics[field]
    return (
        f"| {name} | {values['p50']:.1f} | {values['p90']:.1f} | "
        f"{values['p95']:.1f} | {values['p99']:.1f} | {values['max']} |"
    )


benchmark_lines = [
    f"| {row['batch_size']} | {row['seconds']:.3f} | {row['completed']} | {row['error_type'] or ''} |"
    for row in dense_cache_manifest["benchmark_results"]
]

report = f"""# SHROOM RAG E5 — Module 2 Phase A Results

## Κατάσταση και scope

Η Phase A ολοκληρώθηκε μόνο ως offline candidate retrieval για
**unlabeled-training-corpus RAG**. Δεν χρησιμοποιήθηκαν hallucination labels,
official test data, Gemini/API, relevance thresholds ή tuned fusion weights.

Η `Q_SRC_TGT` είναι reference-assisted retrieval και όχι ανεξάρτητη external
validation.

## Root cause και επιδιόρθωση

Το αρχικό Run All κωδικοποιούσε 9.577 πλήρη concatenated candidate contexts σε
ένα μη-resumable cell. Σε CPU δεν ολοκληρώθηκε μέσα σε 1.200 δευτερόλεπτα και
δεν μπορούσε να διατηρήσει μερική πρόοδο.

Αλλαγές:

- Cells 1–2: τεκμηρίωση bounded/resumable dense retrieval και Q_SRC_TGT limitation.
- Cell 3: offline/CPU deterministic settings πριν από Hugging Face imports.
- Cell 5: cache/helper paths και σωστή απομόνωση του report στο Module 2 output.
- Cell 8: token-aware bounded `Q_SRC`/`Q_SRC_TGT` με diagnostics.
- Cell 10: BM25 με τα ίδια bounded query strings.
- Cell 11: compatibility-checked cache load και encoding μόνο 40 queries.
- Cell 16: cache, shape, normalization, deterministic και leakage tests.
- Cell 17: atomic artifacts, πλήρες manifest και ενημερωμένο results report.
- `scripts/module2_dense_cache.py`: benchmark, atomic chunks και deterministic resume.

## Environment και dense configuration

- Environment: `shroom_ensemble`
- Python: `{os.sys.version.split()[0]}`
- `rank-bm25`: `{version('rank-bm25')}`
- Model: `{EMBEDDING_MODEL_ID}` (local-only, no download)
- Device: `{EMBEDDING_DEVICE}`
- Dimension: `{EMBEDDING_DIMENSION}`
- Normalization/dtype: `{NORMALIZE_EMBEDDINGS}` / `float32`
- Dense text version: `{DENSE_TEXT_VERSION}`
- `max_seq_length`: `{DENSE_MAX_SEQ_LENGTH}`
- Selected batch size: `{selected_batch_size}`
- Chunk size: `{DENSE_CHUNK_SIZE}`

## Token-length statistics

| Inputs | p50 | p90 | p95 | p99 | max |
|---|---:|---:|---:|---:|---:|
{stats_line('Candidates unbounded', CANDIDATE_TOKEN_DIAGNOSTICS, 'unbounded_token_lengths')}
{stats_line('Candidates bounded', CANDIDATE_TOKEN_DIAGNOSTICS, 'bounded_token_lengths')}
{stats_line('Queries unbounded', QUERY_TOKEN_DIAGNOSTICS, 'unbounded_token_lengths')}
{stats_line('Queries bounded', QUERY_TOKEN_DIAGNOSTICS, 'bounded_token_lengths')}

- Candidate context-only truncations: `{CANDIDATE_TOKEN_DIAGNOSTICS['context_only_truncated_count']}`
  (`{CANDIDATE_TOKEN_DIAGNOSTICS['context_only_truncated_percent']:.2f}%`)
- Candidate definition truncations: `{CANDIDATE_TOKEN_DIAGNOSTICS['definition_truncated_count']}`
- Candidate marker/term/initial fallbacks: `{CANDIDATE_TOKEN_DIAGNOSTICS['context_fallback_counts']}`
- Query context-only truncations: `{QUERY_TOKEN_DIAGNOSTICS['context_only_truncated_count']}`
- Query primary-reference truncations: `{QUERY_TOKEN_DIAGNOSTICS['reference_truncated_count']}`
- Query marker/term/initial fallbacks: `{QUERY_TOKEN_DIAGNOSTICS['context_fallback_counts']}`
- Όλα τα bounded inputs έως 256 tokens: `True`

## Fixed 128-candidate benchmark

| Batch size | Seconds | Completed | Error type |
|---:|---:|---|---|
{chr(10).join(benchmark_lines)}

Η επιλογή batch size έγινε μόνο με χρόνο/memory/error στο fixed row-order/seed
sample και όχι με relevance ή hallucination labels.

## Dense cache

- Builder launches: `{dense_cache_manifest['builder_launches']}`
- Resumptions: `{dense_cache_manifest['cache_resumptions']}`
- Completed chunks: `{len(dense_cache_manifest['completed_chunk_ranges'])}`
- Total chunk encoding time: `{dense_cache_manifest['total_chunk_encoding_seconds']:.3f}` seconds
- Last-launch cache hits: `{dense_cache_manifest['last_launch_cache_hits']}`
- Cache complete and compatibility-verified: `True`

## Inputs και artifacts

- DM knowledge-base rows: `{len(dm_kb)}`
- Fixed development sample: `{len(prototype_df)}/20`
- Distinct bounded queries: `{len(query_df)}/40`
- Knowledge-base SHA-256: `{KB_HASH}`
- Sample SHA-256: `{SAMPLE_HASH}`
- Manifest rows: `1`
- Candidate JSONL rows: `{len(candidate_df)}`
- Annotation template rows: `{len(annotation_df)}`
- Manual annotations filled: `0`

## Candidate counts

| Query variant | Retrieval method | Candidate rows |
|---|---|---:|
{chr(10).join(count_lines)}

- Exact non-empty coverage for `Q_SRC`: `{exact_coverage['Q_SRC']:.1%}`
- Exact non-empty coverage for `Q_SRC_TGT`: `{exact_coverage['Q_SRC_TGT']:.1%}`

Οι παραπάνω αριθμοί είναι candidate/coverage diagnostics και όχι relevance
metrics. Δεν υπολογίζονται Precision@k, Relevant@1 ή usable coverage πριν από
manual annotation.

## Regression previews

Τα regression cases προέρχονται από το Module 1 και δεν αποτελούν ανεξάρτητο
evaluation set. Δεν αποδίδεται αυτόματη relevance κρίση.

| Index | Diagnostic term | Query variant | Rank | Evidence ID | Candidate term | Methods | Definition preview |
|---:|---|---|---:|---:|---|---|---|
{chr(10).join(preview_lines)}

## Offline και leakage verification

- Tests passed: `{sum(TEST_SUMMARY.values())}/{len(TEST_SUMMARY)}`
- Cache embeddings: `{candidate_embeddings.shape}`, normalized float32, finite
- Query embeddings: `{query_embeddings.shape}`, normalized float32, finite
- Duplicate candidates μέσα σε ranked list: `0`
- Query variants distinct: `True`
- Forbidden label/probability columns: `0`
- Official test used: `False`
- Gemini/API/credential operations: `0`
- Downloads: `0`
- Module 1 hashes unchanged: `True`

## Limitations και επόμενα βήματα

- Δεν υπάρχουν ακόμη manual relevance annotations.
- Δεν επιλέγεται καλύτερος retriever και δεν ορίζεται relevance gate/threshold.
- Η `Q_SRC_TGT` χρησιμοποιεί την primary reference και δεν αποτελεί ανεξάρτητη
  εξωτερική επιβεβαίωση.
- Τα πέντε regression cases είναι diagnostics, όχι γενίκευση.

Επόμενο επιτρεπτό βήμα είναι μόνο η ξεχωριστά εγκεκριμένη manual συμπλήρωση του
annotation template και ο προκαθορισμός retrieval-development/holdout rows πριν
από οποιοδήποτε tuning ή relevance metric.
"""

atomic_write_text(RESULTS_REPORT_PATH, report)

print("Saved manifest:", MANIFEST_PATH)
print("Saved candidates:", CANDIDATES_PATH)
print("Saved annotations:", ANNOTATIONS_PATH)
print("Saved report:", RESULTS_REPORT_PATH)
print("API calls: 0")